# Stage C: Counterfactual Decoder Training (Colab)

**UltraBERT-Gen MoE Decoder Training Pipeline**

This notebook trains the 13th head - the Counterfactual Decoder with Mixture-of-Experts architecture.

**Prerequisites:**
- Stage B checkpoint (`outputs/modernbert-v2-for-v3-transfer`)
- Counterfactual training data (`data/counterfactual/training`)

**Features:**
- Automatic resume after Colab 18-hour timeout
- Checkpoints saved every 500 steps (~30 min)
- Signal handling for graceful shutdown
- Google Drive persistence

## 1. Environment Setup

In [ ]:
# Check Python and PyTorch versions
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)

In [ ]:
# Install Flash Attention (pre-built wheel for Colab)
!pip install https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.5.4/flash_attn-2.6.3+cu124torch2.9-cp312-cp312-linux_x86_64.whl

In [ ]:
# Verify Flash Attention installation
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Flash SDP Enabled: {torch.backends.cuda.flash_sdp_enabled()}")

import flash_attn
print(f"Flash Attention Version: {flash_attn.__version__}")

In [ ]:
# Check GPU
!nvidia-smi

# Check if we're on Colab
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
print(f"\nRunning on Colab: {IN_COLAB}")

## 2. Repository Setup

In [ ]:
# Clone repository (if on Colab)
import os

REPO_URL = "https://github.com/Pkansagra-hub/Family_osModernBERT.git"
REPO_DIR = "Modeling_studio"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print("Repository already exists, pulling latest...")
        !cd {REPO_DIR} && git pull

    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")
else:
    # Local development - assume we're in the repo root
    print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
print("Installing dependencies...")
!pip install -q -e .
!pip install -q wandb tensorboard h5py sacrebleu
!pip install -e . 'datasets>=2.14.0,<3.0.0' -q

# Install familyos_ultrabert wheel (required for embedding generation)
!pip install -q https://github.com/Pkansagra-hub/Family_osModernBERT/releases/download/v2.2.1/familyos_ultrabert-2.2.1-py3-none-any.whl

print("Dependencies installed!")

In [ ]:
# Mount Google Drive for persistent storage
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Create directories on Drive
    DRIVE_BASE = "/content/drive/MyDrive/FamilyOS_ModernBERT"
    !mkdir -p "{DRIVE_BASE}/data"
    !mkdir -p "{DRIVE_BASE}/outputs"
    !mkdir -p "{DRIVE_BASE}/checkpoints"
    !mkdir -p "{DRIVE_BASE}/data/counterfactual"

    # Symlink outputs to Drive for persistence
    !rm -rf outputs checkpoints 2>/dev/null
    !ln -s "{DRIVE_BASE}/outputs" outputs
    !ln -s "{DRIVE_BASE}/checkpoints" checkpoints

    print(f"Data/outputs will be saved to: {DRIVE_BASE}")
else:
    print("Local mode - outputs saved to local directory")

In [ ]:
# Set environment variables
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow warnings
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Avoid tokenizer warnings

## 3. Data Verification

In [ ]:
# Verify Stage C prerequisites
import os
from pathlib import Path

def check_stage_c_prerequisites():
    """Verify all prerequisites for Stage C training."""

    print("=" * 60)
    print("STAGE C PREREQUISITES CHECK")
    print("=" * 60)

    # Required: Stage B checkpoint
    stage_b_paths = [
        "outputs/modernbert-v2-for-v3-transfer",
        "outputs/modernbert-v2-for-v3-transfer/checkpoint-18000",
        "checkpoints/modernbert-v2-for-v3-transfer",
    ]

    stage_b_ok = False
    stage_b_path = None
    print("\n[1] Stage B Checkpoint (Required):")
    for path in stage_b_paths:
        if os.path.exists(path):
            # Check for model file
            model_file = os.path.join(path, "pytorch_model.bin")
            safetensors_file = os.path.join(path, "model.safetensors")
            if os.path.exists(model_file) or os.path.exists(safetensors_file):
                stage_b_ok = True
                stage_b_path = path
                print(f"    [OK] Found: {path}")
                break

    if not stage_b_ok:
        print("    [MISSING] Stage B checkpoint not found!")
        print("    Expected at: outputs/modernbert-v2-for-v3-transfer")

    # Check: Synthetic JSONL data (raw input for embedding generation)
    synthetic_paths = [
        "data/counterfactual/synthetic",
        "/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/synthetic",
    ]

    synthetic_ok = False
    synthetic_path = None
    print("\n[2] Synthetic JSONL Data (for embedding generation):")
    for path in synthetic_paths:
        if os.path.exists(path):
            jsonl_files = [f for f in os.listdir(path) if f.endswith('.jsonl')]
            if jsonl_files:
                synthetic_ok = True
                synthetic_path = path
                total_samples = 0
                for f in jsonl_files:
                    with open(os.path.join(path, f)) as fp:
                        total_samples += sum(1 for _ in fp)
                print(f"    [OK] Found: {path}")
                print(f"         Shards: {len(jsonl_files)}")
                print(f"         Total samples: ~{total_samples:,}")
                break

    if not synthetic_ok:
        print("    [MISSING] No JSONL shards found!")
        print("    Upload to: data/counterfactual/synthetic/")

    # Check: Prepared training data with full sequence embeddings
    training_paths = [
        "data/counterfactual/training",
        "/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/training",
    ]

    training_ok = False
    training_path = None
    print("\n[3] Prepared Training Data (with embeddings):")
    for path in training_paths:
        if os.path.exists(path):
            samples_file = os.path.join(path, "samples.jsonl")
            seq_embeddings = os.path.join(path, "sequence_embeddings.h5")
            pooled_embeddings = os.path.join(path, "embeddings.h5")

            if os.path.exists(samples_file):
                training_path = path

                # Count samples
                with open(samples_file) as f:
                    num_samples = sum(1 for _ in f)

                has_seq_emb = os.path.exists(seq_embeddings)
                has_pooled_emb = os.path.exists(pooled_embeddings)

                if has_seq_emb:
                    training_ok = True
                    emb_size_gb = os.path.getsize(seq_embeddings) / (1024**3)
                    print(f"    [OK] Found: {path}")
                    print(f"         Samples: {num_samples:,}")
                    print(f"         Full Sequence Embeddings: {emb_size_gb:.2f} GB")
                elif has_pooled_emb:
                    print(f"    [PARTIAL] Found: {path}")
                    print(f"         Samples: {num_samples:,}")
                    print(f"         Pooled Embeddings: Yes")
                    print(f"         Full Sequence: NO - Run section 3.5 to generate!")
                else:
                    print(f"    [PARTIAL] Found samples but no embeddings")
                    print(f"         Run section 3.5 to generate embeddings!")
                break

    if not training_ok and not training_path:
        print("    [NOT READY] Training data not prepared yet")
        print("    Run section 3.5 to generate full sequence embeddings")

    # Summary
    print("\n" + "=" * 60)
    if stage_b_ok and training_ok:
        print("[READY] All prerequisites met! Ready for Stage C training.")
        return True, stage_b_path, training_path
    elif stage_b_ok and synthetic_ok and not training_ok:
        print("[ACTION NEEDED] Run section 3.5 to generate embeddings first!")
        return False, stage_b_path, synthetic_path
    else:
        print("[ERROR] Prerequisites missing. Fix issues above before training.")
        return False, stage_b_path, None
    print("=" * 60)

ready, stage_b_path, data_path = check_stage_c_prerequisites()

## 3.5 Generate Full Sequence Embeddings (GPU)

This step computes encoder embeddings for all counterfactual samples using the A100 GPU.
- **Input**: JSONL files from `data/counterfactual/synthetic/`
- **Output**: HDF5 file with full sequence embeddings (for cross-attention)
- **Time**: ~5-10 min for 100K samples on A100

In [ ]:
%%time
# ============================================================
# GENERATE FULL SEQUENCE EMBEDDINGS ON A100 GPU
# ============================================================
# This converts JSONL counterfactual data → HDF5 embeddings
# Full sequence embeddings enable cross-attention in decoder
#
# NOTE: Output is redirected to log file to prevent Chrome
# from consuming 9GB+ RAM with verbose progress updates!
# ============================================================

import os
from pathlib import Path

# Configuration
SYNTHETIC_DIR = "data/counterfactual/merged"  # Input: JSONL shards
OUTPUT_DIR = "data/counterfactual/training"       # Output: HDF5 + samples
MODEL_PATH = "outputs/modernbert-v2-for-v3-transfer/checkpoint-18000"

# A100-optimized settings
BATCH_SIZE = 128      # Large batch for A100 (reduce to 64 for V100/T4)
MAX_LENGTH = 256      # Max input sequence length
MAX_SAMPLES = ""      # Set to "--max-samples 1000" to limit samples for testing

print("=" * 60)
print("FULL SEQUENCE EMBEDDING GENERATION")
print("=" * 60)

# Check for Drive symlink (Colab)
if os.path.exists("/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/merged"):
    SYNTHETIC_DIR = "/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/merged"
    OUTPUT_DIR = "/content/drive/MyDrive/FamilyOS_ModernBERT/data/counterfactual/training"
    print(f"Using Google Drive paths")

# Check if synthetic data exists
if not os.path.exists(SYNTHETIC_DIR):
    print(f"\n[ERROR] Synthetic data not found at {SYNTHETIC_DIR}")
    print("Upload your JSONL shards to Google Drive first!")
    print("\nExpected structure:")
    print("  data/counterfactual/merged/")
    print("    shard_0000.jsonl")
    print("    shard_0001.jsonl")
    print("    ...")
else:
    # Count JSONL files and samples
    jsonl_files = sorted([f for f in os.listdir(SYNTHETIC_DIR) if f.endswith('.jsonl') and f.startswith('shard')])
    total_samples = 0
    for f in jsonl_files:
        with open(os.path.join(SYNTHETIC_DIR, f)) as fp:
            total_samples += sum(1 for line in fp if line.strip())

    print(f"\nInput: {SYNTHETIC_DIR}")
    print(f"  Shards: {len(jsonl_files)}")
    print(f"  Total samples: {total_samples:,}")

    # Check if embeddings already exist
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    embeddings_file = os.path.join(OUTPUT_DIR, "sequence_embeddings.h5")

    if os.path.exists(embeddings_file):
        size_gb = os.path.getsize(embeddings_file) / (1024**3)
        print(f"\n[INFO] Embeddings already exist!")
        print(f"  Path: {embeddings_file}")
        print(f"  Size: {size_gb:.2f} GB")
        print("\nTo regenerate, delete the file first:")
        print(f"  !rm {embeddings_file}")
    else:
        print(f"\nOutput: {OUTPUT_DIR}")
        print(f"  Batch size: {BATCH_SIZE}")
        print(f"  Max length: {MAX_LENGTH}")

        # Estimate time
        est_time = total_samples / 20000  # ~20K samples/min on A100
        print(f"\nEstimated time: ~{est_time:.0f} min")
        print("=" * 60)
        print("\nStarting embedding generation...")
        print("Output redirected to embedding_gen.log to save browser RAM")
        print("Check GPU usage: watch nvidia-smi in another cell")
        print("=" * 60)

        # Run with output redirected to log file (prevents Chrome RAM bloat)
        LOG_FILE = os.path.join(OUTPUT_DIR, "embedding_gen.log")
        !python scripts/agents/prepare_decoder_training_data.py \
            --input-dir "{SYNTHETIC_DIR}" \
            --output-dir "{OUTPUT_DIR}" \
            --model-path "{MODEL_PATH}" \
            --full-sequence \
            --batch-size {BATCH_SIZE} \
            --max-length {MAX_LENGTH} \
            --device cuda \
            {MAX_SAMPLES} > "{LOG_FILE}" 2>&1

        # Show last 20 lines of log
        print("\n--- Last 20 lines of log ---")
        !tail -20 "{LOG_FILE}"

        # Verify result
        if os.path.exists(embeddings_file):
            print("\n" + "=" * 60)
            print("[SUCCESS] Full sequence embeddings generated!")
            print("=" * 60)

            # Show output files
            print(f"\nOutput files in {OUTPUT_DIR}:")
            for f in sorted(os.listdir(OUTPUT_DIR)):
                fpath = os.path.join(OUTPUT_DIR, f)
                if os.path.isfile(fpath):
                    size_mb = os.path.getsize(fpath) / (1024 * 1024)
                    print(f"  {f}: {size_mb:.1f} MB")

            # Verify embeddings
            import h5py
            with h5py.File(embeddings_file, 'r') as hf:
                num_tokens = hf['embeddings'].shape[0]
                hidden_dim = hf['embeddings'].shape[1]
                num_samples = hf.attrs.get('num_samples', 'N/A')
                print(f"\nEmbedding stats:")
                print(f"  Total tokens: {num_tokens:,}")
                print(f"  Hidden dim: {hidden_dim}")
                print(f"  Samples: {num_samples:,}")
        else:
            print(f"\n[ERROR] Embedding generation failed!")
            print(f"Check full log: !cat {LOG_FILE}")

## 4. Stage C Training

**Important Notes:**
- Training saves checkpoints every 500 steps (~30 min)
- If Colab disconnects (18-hour limit), use the "Resume Training" cell below
- `--auto_resume` automatically finds the latest checkpoint

In [ ]:
# Pull latest code before training
!cd /content/Modeling_studio && git pull origin main

In [ ]:
%%time

# ============================================================
# STAGE C: COUNTERFACTUAL DECODER TRAINING (FRESH START)
# ============================================================
# Use this cell for FIRST training run
# For resuming after disconnect, use the next cell instead

import os

print("=" * 60)
print("STAGE C: COUNTERFACTUAL DECODER TRAINING")
print("=" * 60)
print("  - Decoder: 8 layers, MoE with 8 experts")
print("  - Encoder: FROZEN (Stage B checkpoint)")
print("  - Checkpoints: Every 500 steps")
print("=" * 60)

# Run Stage C training
!python scripts/train_stage_c.py \
    --config configs/training/multitask/stage_c_decoder.yaml

# Check if training succeeded
output_dir = "outputs/ultrabert-gen-decoder-v1"
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    has_model = any(f.endswith(('.bin', '.safetensors')) for f in files)
    if has_model:
        print("\n" + "=" * 60)
        print("[SUCCESS] STAGE C COMPLETED!")
        print("=" * 60)
    else:
        print("\n" + "=" * 60)
        print("[INFO] Training in progress - checkpoints saved")
        print("=" * 60)
else:
    print("\n" + "=" * 60)
    print("[ERROR] STAGE C FAILED! Check logs above.")
    print("=" * 60)

### Resume Training (After Colab Disconnect)

**Use this cell if:**
- Colab disconnected due to 18-hour timeout
- You manually stopped training
- Training crashed and you want to continue

The `--auto_resume` flag automatically finds the latest checkpoint.

In [ ]:
%%time

# ============================================================
# RESUME STAGE C TRAINING (AFTER DISCONNECT)
# ============================================================
# Use this cell to RESUME training from latest checkpoint
# --auto_resume automatically finds checkpoint-XXXX folder

import os

print("=" * 60)
print("RESUMING STAGE C TRAINING")
print("=" * 60)

# Check for existing checkpoints
output_dir = "outputs/ultrabert-gen-decoder-v1"
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        print(f"  Found {len(checkpoints)} checkpoint(s)")
        print(f"  Latest: {latest}")
    else:
        print("  No checkpoints found - will start fresh")
else:
    print("  No previous training found - will start fresh")

print("=" * 60)

# Resume training with --auto_resume
!python scripts/train_stage_c.py \
    --config configs/training/multitask/stage_c_decoder.yaml \
    --auto_resume

# Check completion
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    has_model = any(f.endswith(('.bin', '.safetensors')) for f in files)
    if has_model:
        print("\n" + "=" * 60)
        print("[SUCCESS] STAGE C COMPLETED!")
        print("=" * 60)

## 5. Verify Training Output

In [ ]:
# Verify Stage C output
import os
import json

stage_c_output = "outputs/ultrabert-gen-decoder-v1"

if os.path.exists(stage_c_output):
    print(f"Stage C output: {stage_c_output}")
    print("\nFiles:")

    total_size = 0
    for f in sorted(os.listdir(stage_c_output)):
        fpath = os.path.join(stage_c_output, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1e6
            total_size += size
            print(f"    {f} ({size:.1f} MB)")
        elif os.path.isdir(fpath):
            print(f"    {f}/ (checkpoint)")

    print(f"\nTotal size: {total_size:.1f} MB")

    # Load training state if available
    trainer_state = os.path.join(stage_c_output, "trainer_state.json")
    if os.path.exists(trainer_state):
        with open(trainer_state) as f:
            state = json.load(f)
        print(f"\nTraining Progress:")
        print(f"    Global step: {state.get('global_step', 'N/A')}")
        print(f"    Epoch: {state.get('epoch', 'N/A'):.2f}")

        # Get best metric if available
        if 'best_metric' in state:
            print(f"    Best loss: {state['best_metric']:.4f}")

    # Load eval results
    eval_path = os.path.join(stage_c_output, "eval_results.json")
    if os.path.exists(eval_path):
        with open(eval_path) as f:
            results = json.load(f)
        print("\nEval Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"    {k}: {v:.4f}")
else:
    print(f"[INFO] Stage C output not found at {stage_c_output}")
    print("       Training may still be in progress.")

In [ ]:
# Check decoder parameter count
import torch

def count_parameters(checkpoint_path):
    """Count parameters in the decoder head."""
    try:
        # Try loading the state dict
        for fname in ["pytorch_model.bin", "model.safetensors"]:
            fpath = os.path.join(checkpoint_path, fname)
            if os.path.exists(fpath):
                if fname.endswith(".bin"):
                    state_dict = torch.load(fpath, map_location="cpu")
                else:
                    from safetensors.torch import load_file
                    state_dict = load_file(fpath)

                # Count decoder parameters
                decoder_params = 0
                encoder_params = 0
                head_params = 0

                for name, param in state_dict.items():
                    num_params = param.numel()
                    if "counterfactual" in name.lower() or "decoder" in name.lower():
                        decoder_params += num_params
                    elif "encoder" in name.lower():
                        encoder_params += num_params
                    else:
                        head_params += num_params

                total = decoder_params + encoder_params + head_params
                print(f"\nParameter Count:")
                print(f"    Encoder (frozen): {encoder_params / 1e6:.1f}M")
                print(f"    Existing heads:   {head_params / 1e6:.1f}M")
                print(f"    Decoder (new):    {decoder_params / 1e6:.1f}M")
                print(f"    Total:            {total / 1e6:.1f}M")
                return

        print("No model file found in checkpoint")
    except Exception as e:
        print(f"Could not load model: {e}")

if os.path.exists(stage_c_output):
    count_parameters(stage_c_output)

## 6. Training Summary & Backup

In [ ]:
# Training Summary
import os
from datetime import datetime

print("=" * 60)
print("STAGE C TRAINING SUMMARY")
print("=" * 60)
print(f"   Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check outputs
stage_b_ok = os.path.exists("outputs/modernbert-v2-for-v3-transfer")
stage_c_output = "outputs/ultrabert-gen-decoder-v1"
stage_c_ok = os.path.exists(stage_c_output)

# Check for final model
stage_c_complete = False
if stage_c_ok:
    files = os.listdir(stage_c_output)
    stage_c_complete = any(f.endswith(('.bin', '.safetensors')) and not f.startswith('checkpoint') for f in files)

print(f"\n   Stage B (base model): {'[OK]' if stage_b_ok else '[MISSING]'}")
print(f"   Stage C (decoder):    {'[COMPLETE]' if stage_c_complete else '[IN PROGRESS]' if stage_c_ok else '[NOT STARTED]'}")

if stage_c_ok:
    # Count checkpoints
    checkpoints = [d for d in os.listdir(stage_c_output) if d.startswith("checkpoint-")]
    print(f"\n   Checkpoints saved: {len(checkpoints)}")
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        step = int(latest.split("-")[1])
        print(f"   Latest checkpoint: {latest} (step {step})")

print("\n" + "=" * 60)
print("OUTPUT LOCATIONS")
print("=" * 60)

if stage_b_ok:
    print("   Stage B: outputs/modernbert-v2-for-v3-transfer")
    print("            -> 12-head encoder (frozen during Stage C)")

if stage_c_ok:
    print("   Stage C: outputs/ultrabert-gen-decoder-v1")
    print("            -> 13th head: Counterfactual MoE Decoder")
    print("            -> ~420M parameters (decoder only)")

print("\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
if stage_c_complete:
    print("   1. Run evaluation metrics (Milestone 15)")
    print("   2. Test generation quality with sample inputs")
    print("   3. Deploy model for inference")
else:
    print("   1. Wait for training to complete OR")
    print("   2. Resume training if disconnected (use Resume cell)")
    print("   3. Run evaluation after training completes")
print("=" * 60)

In [ ]:
# Backup outputs to Google Drive (with timestamp)
if IN_COLAB:
    import shutil
    from datetime import datetime

    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    backup_dir = f"/content/drive/MyDrive/FamilyOS_ModernBERT/runs/stage_c_{timestamp}"

    print(f"Creating backup at: {backup_dir}")
    os.makedirs(backup_dir, exist_ok=True)

    # Copy Stage C output
    stage_c_output = "outputs/ultrabert-gen-decoder-v1"
    if os.path.exists(stage_c_output):
        shutil.copytree(
            stage_c_output,
            f"{backup_dir}/ultrabert-gen-decoder-v1",
            dirs_exist_ok=True
        )
        print("   [OK] Stage C output backed up")

        # Calculate size
        total_size = 0
        for root, dirs, files in os.walk(f"{backup_dir}/ultrabert-gen-decoder-v1"):
            for f in files:
                total_size += os.path.getsize(os.path.join(root, f))
        print(f"   Backup size: {total_size / 1e9:.2f} GB")
    else:
        print("   [SKIP] No Stage C output to backup")

    print(f"\nBackup complete: {backup_dir}")
else:
    print("Not on Colab - backup to Drive skipped")

## 7. Test Generation (Optional)

Quick test to verify the decoder generates reasonable output.

In [ ]:
# Quick generation test (run after training completes)
import torch
from transformers import AutoTokenizer

def test_generation():
    """Test the trained decoder with a sample input."""
    try:
        from modeling_studio.models import ModernBertMultiTaskModel
        from modeling_studio.data.labels import Capability

        print("Loading model...")
        model = ModernBertMultiTaskModel.load_checkpoint(
            "outputs/ultrabert-gen-decoder-v1",
            device="cuda" if torch.cuda.is_available() else "cpu"
        )
        model.eval()

        tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

        # Test input
        test_input = "I yelled at my kids this morning and now I feel terrible about it."

        print(f"\nInput: {test_input}")
        print("\nGenerating counterfactual...")

        # Encode input
        inputs = tokenizer(test_input, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Get encoder output
        with torch.no_grad():
            encoder_output = model.encoder(**inputs)
            hidden_states = encoder_output.last_hidden_state

            # Generate with decoder head
            if Capability.COUNTERFACTUAL in model.heads:
                decoder_head = model.heads[Capability.COUNTERFACTUAL]

                # Use pooled embedding (mean pooling)
                attention_mask = inputs['attention_mask']
                pooled = (hidden_states * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1, keepdim=True)

                # Generate
                generated_ids = decoder_head.generate(
                    encoder_hidden_states=pooled.unsqueeze(1),
                    encoder_attention_mask=torch.ones(1, 1, device=model.device),
                    max_length=128,
                    temperature=0.7,
                    top_p=0.9,
                )

                output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                print(f"\nCounterfactual: {output_text}")
            else:
                print("Counterfactual head not found in model")

    except Exception as e:
        print(f"Generation test failed: {e}")
        print("This is expected if training is not complete.")

# Only run if model exists
if os.path.exists("outputs/ultrabert-gen-decoder-v1"):
    test_generation()
else:
    print("Model not found - run training first")